# Mini Projects: CartPole, FrozenLake, Q-learning, and DQN

## 📚 Learning Objectives

By completing this notebook, you will:
- Implement Q-learning algorithm
- Apply Q-learning to FrozenLake environment
- Apply Q-learning to CartPole environment (with state discretization)
- Understand Deep Q-Network (DQN) concepts
- Train and evaluate RL agents on classic environments

## 🔗 Prerequisites

- ✅ OpenAI Gym setup
- ✅ Understanding of states, actions, rewards
- ✅ Epsilon-Greedy exploration strategy
- ✅ Python knowledge (functions, classes, loops, dictionaries)
- ✅ NumPy knowledge
- ✅ Basic understanding of neural networks (for DQN section)

---

## Official Structure Reference

This notebook covers practical activities from **Course 09, Unit 1**:
- Mini projects: applying RL in games like CartPole and FrozenLake, implementing Q-learning and DQN
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 1 Practical Content

---

## Introduction

This notebook combines multiple mini projects:
1. **Q-learning on FrozenLake**: Classic grid-world problem with discrete states
2. **Q-learning on CartPole**: Continuous state space requiring discretization
3. **DQN Introduction**: Deep reinforcement learning for high-dimensional states

These projects demonstrate practical RL applications on classic benchmark environments.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
%pip install gymnasium[classic_control] numpy matplotlib -q
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import random
from collections import defaultdict

print("✅ Libraries imported!")
print("\nMini Projects: CartPole, FrozenLake, Q-learning, and DQN")
print("=" * 60)

## Part 1: Q-Learning Algorithm Implementation


In [ ]:
print("=" * 60)
print("Part 1: Q-Learning Algorithm Implementation")
print("=" * 60)


## Part 2: Q-Learning on FrozenLake


In [ ]:
print("\n" + "=" * 60)
print("Part 2: Q-Learning on FrozenLake")
print("=" * 60)


## Part 3: Q-Learning on CartPole (with State Discretization)


In [ ]:
print("\n" + "=" * 60)
print("Part 3: Q-Learning on CartPole (with State Discretization)")
print("=" * 60)


## Part 4: Introduction to Deep Q-Network (DQN)


In [ ]:
print("\n" + "=" * 60)
print("Part 4: Introduction to Deep Q-Network (DQN)")
print("=" * 60)

print("\nDQN Overview:")
print(" - Uses neural network to approximate Q-function (instead of Q-table)")
print(" - Handles high-dimensional state spaces (e.g., images)")
print(" - Key innovations:")
print(" 1. Experience Replay: Store transitions, sample randomly for training")
print(" 2. Target Network: Separate network for stable Q-targets")
print(" 3. Neural Network: Approximate Q(s,a) for continuous/ high-dim states")

print("\nDQN Architecture:")
print(" Input: State (e.g., image, observation vector)")
print(" Network: Fully connected or CNN layers")
print(" Output: Q-values for each action")

print("\nDQN vs Q-Learning:")
print(" Q-Learning:")
print(" - Uses Q-table (discrete states only)")
print(" - Limited to small state spaces")
print(" - Fast for tabular problems")
print(" DQN:")
print(" - Uses neural network (continuous/high-dim states)")
print(" - Scales to complex problems (e.g., Atari games)")
print(" - Requires more computation and tuning")

print("\nDQN Algorithm (High-Level):")
print(" 1. Initialize Q-network and target network")
print(" 2. For each episode:")
print(" a. Observe state")
print(" b. Choose action using epsilon-greedy (using Q-network)")
print(" c. Take action, store transition in replay buffer")
print(" d. Sample batch from replay buffer")
print(" e. Compute targets using target network")
print(" f. Update Q-network using loss: (Q(s,a) - target)^2")
print(" g. Periodically update target network")

print("\nNote: Full DQN implementation will be covered in Unit 3 (Deep RL)")
print("This is an introduction to the concepts.")

print("\n✅ DQN introduction complete!")

## 🌍 Real-World Worked Example — CartPole with DQN

**Industry context:**
- Boston Dynamics uses policy gradient variants (similar to DQN) for robot balance control
- Tesla's Autopilot reward signal includes smooth lane-keeping (like CartPole balance)

We train a **DQN agent** to balance a pole on a cart using raw observations from OpenAI Gymnasium.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import gymnasium as gym
import numpy as np, random, collections, matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)

env = gym.make('CartPole-v1')

# ── DQN Network ──────────────────────────────────────────────────────────
class DQN(nn.Module):
    def __init__(self, obs=4, act=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs,128), nn.ReLU(),
            nn.Linear(128,128), nn.ReLU(),
            nn.Linear(128, act)
        )
    def forward(self, x): return self.net(x)

policy_net = DQN(); target_net = DQN()
target_net.load_state_dict(policy_net.state_dict())
opt     = optim.Adam(policy_net.parameters(), lr=1e-3)
memory  = collections.deque(maxlen=10000)
GAMMA   = 0.99; EPS = 1.0; EPS_MIN = 0.05; EPS_DECAY = 0.995
BATCH   = 64; rewards_ep = []

for episode in range(300):
    obs,_ = env.reset(); total_r = 0
    for t in range(500):
        if random.random() < EPS:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = policy_net(torch.tensor(obs).float().unsqueeze(0)).argmax().item()
        obs2, r, done, trunc, _ = env.step(action)
        memory.append((obs, action, r, obs2, done or trunc))
        obs = obs2; total_r += r
        # ── Train ────────────────────────────────────────────────────────
        if len(memory) >= BATCH:
            batch = random.sample(memory, BATCH)
            s,a,r_b,s2,d = zip(*batch)
            S=torch.tensor(np.array(s)).float(); A=torch.tensor(a).long()
            R=torch.tensor(r_b).float(); S2=torch.tensor(np.array(s2)).float()
            D=torch.tensor(d).float()
            Q_pred = policy_net(S).gather(1,A.unsqueeze(1)).squeeze()
            with torch.no_grad():
                Q_next = target_net(S2).max(1)[0]
            Q_target = R + GAMMA*Q_next*(1-D)
            loss = nn.MSELoss()(Q_pred, Q_target)
            opt.zero_grad(); loss.backward(); opt.step()
        if done or trunc: break
    EPS = max(EPS*EPS_DECAY, EPS_MIN)
    rewards_ep.append(total_r)
    if episode%50==0: target_net.load_state_dict(policy_net.state_dict())
    if episode%30==0: print(f"Episode {episode:3d} | Avg reward (last 30): {np.mean(rewards_ep[-30:]):.1f} | ε={EPS:.3f}")

env.close()
plt.plot(rewards_ep, alpha=0.4, label="Episode reward")
plt.plot(np.convolve(rewards_ep, np.ones(20)/20, 'valid'), label="20-ep avg", lw=2)
plt.title("DQN on CartPole-v1"); plt.xlabel("Episode"); plt.ylabel("Reward")
plt.axhline(475, color='red', linestyle='--', label="Solved (475)"); plt.legend()
plt.tight_layout(); plt.show()

## Summary

### Key Concepts:
1. **Q-Learning**: Off-policy TD learning algorithm
   - Updates: Q(s,a) = Q(s,a) + α[r + γ*max(Q(s',a')) - Q(s,a)]
   - Uses Q-table for discrete states
   - Epsilon-greedy exploration

2. **FrozenLake**: Discrete grid-world environment
   - Perfect for tabular Q-learning
   - 16 states, 4 actions
   - Slippery/unslippery variants

3. **CartPole**: Continuous state space
   - Requires state discretization for Q-learning
   - 4 continuous state variables → discrete bins
   - Alternative: Use DQN for continuous states

4. **DQN**: Deep Q-Network
   - Neural network approximates Q-function
   - Handles high-dimensional/continuous states
   - Experience replay and target networks for stability

### Implementation Highlights:
- **Q-Learning**: Tabular method, fast for discrete problems
- **State Discretization**: Convert continuous to discrete for Q-learning
- **Epsilon Decay**: Reduce exploration over time
- **DQN**: Deep learning extension for complex problems

### Best Practices:
- Start with high epsilon (exploration), decay over time
- Tune learning rate (alpha) and discount factor (gamma)
- Monitor learning curves and success rates
- Use experience replay and target networks for DQN stability

### Next Steps:
- Unit 2: Advanced Q-learning (SARSA, TD methods)
- Unit 3: Deep RL (DQN, Actor-Critic, PPO)
- Unit 4: Exploration strategies
- Unit 5: Advanced applications

**Reference:** Course 09, Unit 1: "Introduction to Reinforcement Learning" - Mini projects practical content

## 📚 References & Further Reading

**Papers:**
- Watkins & Dayan (1992) — [Q-Learning](https://link.springer.com/article/10.1007/BF00992698)
- Mnih et al. (2015) — [DQN: Human-level control via deep RL](https://www.nature.com/articles/nature14236)
- Van Hasselt et al. (2016) — [Double DQN](https://arxiv.org/abs/1509.06461)

**OpenAI Gym:** [Gymnasium Documentation](https://gymnasium.farama.org/)

**State-of-the-Art:** DQN variants power game-playing AI and robotics controllers at Google DeepMind.